# PMC Chunks Parquet 数据探索
读取切分好的 chunk 数据，查看内容、统计分布、示例文本。

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', 120)
pd.set_option('display.float_format', '{:.2f}'.format)

## 1. 选择文件

In [ ]:
OUTPUT_DIR = Path(r'd:\Rag-Med\pipeline_output')

# 列出所有可用的 parquet 文件
parquet_files = sorted(OUTPUT_DIR.glob('chunks_*.parquet'))
for i, f in enumerate(parquet_files):
    size_mb = f.stat().st_size / 1e6
    print(f'[{i}] {f.name:<45} {size_mb:>8.1f} MB')

In [ ]:
# 修改这里选择要读取的文件（填上方列表中的索引或直接写路径）
FILE_INDEX = 0

parquet_path = parquet_files[FILE_INDEX]
print(f'读取文件：{parquet_path}')

df = pd.read_parquet(parquet_path)
print(f'行数：{len(df):,}    列数：{len(df.columns)}')

## 2. 基本信息

In [ ]:
# 列名与数据类型
df.dtypes.to_frame('dtype')

In [ ]:
# 空值统计
null_stats = pd.DataFrame({
    'null_count':  df.isnull().sum(),
    'null_pct':    (df.isnull().mean() * 100).round(2),
}).query('null_count > 0')

if null_stats.empty:
    print('无空值')
else:
    display(null_stats)

In [ ]:
# 前 5 行预览（核心字段）
preview_cols = ['chunk_id', 'chunk_type', 'imrad_type', 'section_title',
                'token_count', 'split_strategy', 'journal', 'pub_year']
df[preview_cols].head()

## 3. Token 分布

In [ ]:
tc = df['token_count'].dropna().astype(int)

print('Token 统计')
print(f'  均值   : {tc.mean():.1f}')
print(f'  中位数 : {tc.median():.1f}')
print(f'  p90    : {tc.quantile(0.90):.1f}')
print(f'  p95    : {tc.quantile(0.95):.1f}')
print(f'  p99    : {tc.quantile(0.99):.1f}')
print(f'  最大值 : {tc.max()}')
print(f'  最小值 : {tc.min()}')
print(f'  超 512 : {(tc > 512).sum():,} 块 ({(tc > 512).mean()*100:.2f}%)')

In [ ]:
# Token 分布直方图
bins   = [0, 50, 100, 200, 300, 400, 512, 600, 800, 9999]
labels = ['<50','50-99','100-199','200-299','300-399','400-511','512-599','600-799','>=800']
hist   = pd.cut(tc, bins=bins, labels=labels).value_counts().reindex(labels)

print(f'\nToken 分布直方图（共 {len(tc):,} chunks）\n')
max_cnt = hist.max()
for label, cnt in hist.items():
    bar = '█' * int(cnt / max_cnt * 40)
    print(f'  {label:<10}  {cnt:>8,}  {bar}')

## 4. 分布统计

In [ ]:
# chunk_type / split_strategy / imrad_type 分布
for col in ['chunk_type', 'split_strategy', 'imrad_type']:
    if col not in df.columns:
        continue
    vc = df[col].value_counts()
    pct = (vc / len(df) * 100).round(1)
    print(f'\n── {col} ─────────────────────')
    for k, v in vc.items():
        print(f'  {k:<30} {v:>8,}  ({pct[k]:.1f}%)')

In [ ]:
# 文章数 / 每篇 chunks 均值
n_docs = df['doc_id'].nunique()
print(f'唯一文献数    : {n_docs:,}')
print(f'总 chunk 数   : {len(df):,}')
print(f'chunks / 篇   : {len(df)/n_docs:.1f}')

if 'chunk_type' in df.columns:
    abs_chunks  = (df['chunk_type'] == 'abstract').sum()
    body_chunks = (df['chunk_type'] == 'body').sum()
    print(f'摘要 chunks   : {abs_chunks:,}')
    print(f'正文 chunks   : {body_chunks:,}')

In [ ]:
# 发表年份分布（近 10 年）
if 'pub_year' in df.columns:
    year_dist = (
        df.drop_duplicates('doc_id')['pub_year']
        .dropna().astype(int)
        .value_counts()
        .sort_index(ascending=False)
        .head(15)
    )
    print('发表年份分布（按文献去重，近 15 年）\n')
    max_cnt = year_dist.max()
    for year, cnt in year_dist.items():
        bar = '█' * int(cnt / max_cnt * 30)
        print(f'  {year}  {cnt:>6,}  {bar}')

In [ ]:
# Top 10 期刊
if 'journal' in df.columns:
    print('Top 10 期刊（按文献数）\n')
    top_journals = (
        df.drop_duplicates('doc_id')['journal']
        .value_counts()
        .head(10)
    )
    for j, cnt in top_journals.items():
        print(f'  {cnt:>5,}  {j}')

## 5. 查看具体 Chunk 内容

In [ ]:
def show_chunk(row):
    """格式化打印单个 chunk 的所有字段。"""
    meta_fields = ['chunk_id','doc_id','chunk_type','section_title','section_path',
                   'imrad_type','split_strategy','token_count','chunk_index','total_chunks',
                   'journal','pub_year','article_type','abstract_source','source_url']
    print('─' * 70)
    for f in meta_fields:
        if f in row.index:
            print(f'  {f:<20}: {row[f]}')
    print(f'\n  text ({row["token_count"]} tokens):\n')
    print(row['text'])
    print('─' * 70)

In [ ]:
# 随机查看 3 个 abstract chunk
sample = df[df['chunk_type'] == 'abstract'].sample(3, random_state=42)
for _, row in sample.iterrows():
    show_chunk(row)

In [ ]:
# 随机查看 3 个 body chunk（Methods 类型）
sample = df[(df['chunk_type'] == 'body') & (df['imrad_type'] == 'methods')].sample(
    min(3, len(df[(df['chunk_type'] == 'body') & (df['imrad_type'] == 'methods')])),
    random_state=42
)
for _, row in sample.iterrows():
    show_chunk(row)

In [ ]:
# 查看某篇文献的所有 chunk（输入 doc_id）
DOC_ID = df['doc_id'].iloc[0]   # 改成任意 pmc_id

doc_chunks = df[df['doc_id'] == DOC_ID].sort_values('chunk_index')
print(f'文献 {DOC_ID}：共 {len(doc_chunks)} 个 chunk\n')
print(doc_chunks[['chunk_index','chunk_type','imrad_type','section_title',
                   'token_count','split_strategy']].to_string(index=False))

In [ ]:
# 打印该文献的全部 chunk 文本
for _, row in doc_chunks.iterrows():
    show_chunk(row)

## 6. 批次文件探索（batches 目录）

In [ ]:
# 列出 batches_full 目录状态
batch_dirs = sorted(OUTPUT_DIR.glob('batches_*'))
for bd in batch_dirs:
    files = sorted(bd.glob('batch_*.parquet'))
    if not files:
        continue
    total_size = sum(f.stat().st_size for f in files) / 1e9
    first = int(files[0].stem.split('_')[1])
    last  = int(files[-1].stem.split('_')[1])
    print(f'{bd.name}:')
    print(f'  批次数  : {len(files):,}')
    print(f'  范围    : batch_{first:08d} → batch_{last:08d}')
    print(f'  总大小  : {total_size:.2f} GB')

In [ ]:
# 读取单个批次文件查看内容
BATCH_DIR   = OUTPUT_DIR / 'batches_full'   # 修改为实际目录名
BATCH_INDEX = 0                              # 第几个批次（0=第一个）

batch_files = sorted(BATCH_DIR.glob('batch_*.parquet'))
if batch_files:
    bf = batch_files[BATCH_INDEX]
    print(f'读取：{bf.name}  ({bf.stat().st_size/1e6:.1f} MB)')
    bdf = pd.read_parquet(bf)
    print(f'行数：{len(bdf):,}')
    display(bdf[['chunk_id','chunk_type','imrad_type','section_title',
                 'token_count','journal','pub_year']].head(10))
else:
    print(f'{BATCH_DIR} 目录不存在或无批次文件')